# scWAT Xenium section QC: Phases 0-2

Run this notebook once per section. It retains every cell, preserves raw sparse counts, and adds advisory alarm/gene/spatial diagnostics plus immutable downstream masks.

## Goal

Validate one Xenium section, reconcile its panel, import sparse counts, calculate section-specific QC flags, and write reload-validated core and extended artifact bundles.

## Setup

### Parameters

Change `REGION_ID` for manual execution. Launchers inject the same parameters without editing the source notebook.

In [ ]:
PROJECT_ROOT <- "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium"
PIPELINE_REPO <- file.path(PROJECT_ROOT, "adipose_analysis", "YNH_Xenium_scWAT")
INPUT_ROOT <- file.path(PROJECT_ROOT, "adipose_data")
REGION_ID <- "Region_4"
RUN_LABEL <- "full_notebook_qc_v2"
METADATA_PATH <- file.path(PIPELINE_REPO, "config", "scwat_sample_manifest.tsv")
EXPECTED_SECTION_COUNT <- 4L
SEED <- 20260814L
STRICT_MODE <- FALSE
EXTENDED_QC_MODE <- "AUTO"
EXTENDED_QC_CONFIG_PATH <- file.path(PIPELINE_REPO, "config", "extended_qc_defaults.tsv")


In [ ]:
OUTPUT_ROOT <- file.path(PROJECT_ROOT, "adipose_analysis", "scwat_qc_outputs", RUN_LABEL)
SECTION_OUTPUT_DIR <- file.path(OUTPUT_ROOT, "sections", REGION_ID)
source(file.path(PIPELINE_REPO, "R", "source.R"))
for (package in c("Matrix", "jsonlite", "ggplot2")) require_package(package)
validate_runtime_paths(PROJECT_ROOT, INPUT_ROOT, SECTION_OUTPUT_DIR, tempdir())
dir.create(SECTION_OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
stopifnot(REGION_ID %in% paste0("Region_", seq_len(EXPECTED_SECTION_COUNT)))
cat("Section:", REGION_ID, "\nOutput:", SECTION_OUTPUT_DIR, "\n")


## Inputs

The selected raw/subset section directory is discovered by region ID. All raw files remain read-only.

In [ ]:
all_sections <- discover_xenium_sections(INPUT_ROOT, EXPECTED_SECTION_COUNT)
selected_section <- discover_one_section(INPUT_ROOT, REGION_ID)
selected_section


## Phase 0 - Configuration and metadata contract

In [ ]:
if (nzchar(METADATA_PATH)) {
  assert_path_within(PROJECT_ROOT, METADATA_PATH)
  full_manifest <- utils::read.delim(METADATA_PATH, check.names = FALSE)
} else {
  full_manifest <- create_synthetic_manifest(all_sections$region_id, SEED)
}
manifest_check <- validate_sample_manifest(full_manifest, all_sections$region_id)
if (!manifest_check$valid) stop(paste(manifest_check$issues, collapse = "; "))
section_manifest <- full_manifest[full_manifest$region_id == REGION_ID, , drop = FALSE]
section_manifest$region_dir <- selected_section$region_dir
configuration <- data.frame(
  key = c("project_root", "pipeline_repo", "input_root", "output_root", "region_id", "run_label", "seed", "strict_mode"),
  value = c(PROJECT_ROOT, PIPELINE_REPO, INPUT_ROOT, OUTPUT_ROOT, REGION_ID, RUN_LABEL, SEED, STRICT_MODE)
)
environment <- data.frame(
  item = c("R_version", "platform", "tempdir", "Matrix", "jsonlite", "ggplot2"),
  value = c(R.version.string, R.version$platform, tempdir(), as.character(packageVersion("Matrix")), as.character(packageVersion("jsonlite")), as.character(packageVersion("ggplot2")))
)
section_manifest


## Phase 1 - Provenance, integrity, panel reconciliation, and QC gate

In [ ]:
region_dir <- selected_section$region_dir[[1]]
inventory <- inventory_section_files(region_dir, REGION_ID, calculate_md5 = TRUE)
if (!all(inventory$exists)) stop(paste("Missing required files:", paste(inventory$relative_path[!inventory$exists], collapse = ", ")))
integrity <- validate_section_integrity(region_dir, REGION_ID)
alarms <- extract_analysis_alarms(file.path(region_dir, "analysis_summary.html"))
signature <- utils::read.delim(file.path(PIPELINE_REPO, "config", "eos_gene_sets.tsv"), check.names = FALSE)
installed_genes <- read_custom_panel_genes(file.path(region_dir, "gene_panel.json"))
gene_sets <- setNames(signature$gene_set, signature$gene)
panel_reconciliation <- reconcile_panel(signature$gene, installed_genes, gene_sets)
features_preview <- read_xenium_features(file.path(region_dir, "cell_feature_matrix", "features.tsv.gz"))
feature_type_summary <- as.data.frame(table(features_preview$feature_type), stringsAsFactors = FALSE)
names(feature_type_summary) <- c("feature_type", "n_features")
list(integrity = integrity, alarms = alarms, panel = table(panel_reconciliation$status))


## Phase 2 - Sparse import and section-specific cell QC

In [ ]:
xenium <- import_xenium_mex(region_dir)
qc <- calculate_xenium_cell_qc(xenium$counts, xenium$cells, REGION_ID)
metadata_columns <- intersect(c("mouse_id", "side", "section_id", "biological_replicate_id", "genotype", "treatment", "condition", "age_weeks", "metadata_status", "do_not_interpret"), names(section_manifest))
for (column in metadata_columns) qc$cell_metadata[[column]] <- section_manifest[[column]][[1]]
qc$summary


In [ ]:
section_plots <- plot_section_qc(qc$cell_metadata, REGION_ID)
print(section_plots$counts)
print(section_plots$features)
print(section_plots$area)
print(section_plots$spatial)


## Extended QC preflight and direct alarm evidence

Inputs are the section directory, versioned extended-QC settings, and existing Xenium alarm records. The exact affected cycle remains unresolved without 10x diagnostics.

In [ ]:
extended_config <- read_extended_qc_config(EXTENDED_QC_CONFIG_PATH)
extended_mode <- resolve_extended_qc_mode(EXTENDED_QC_MODE, region_dir)
extended_preflight <- extended_qc_preflight(extended_mode, region_dir, extended_config)
if (extended_mode == "FULL_HPC" && any(extended_preflight$status == "FAIL")) {
  stop(paste("FULL_HPC extended preflight failed:", paste(extended_preflight$check[extended_preflight$status == "FAIL"], collapse = ", ")))
}
cycle_alarm_evidence <- build_cycle_alarm_evidence(alarms, REGION_ID)
cat("Extended mode:", extended_mode, "\nInput:", region_dir, "\nOutput:", SECTION_OUTPUT_DIR, "\n")
extended_preflight
cycle_alarm_evidence


## Extended per-gene quality diagnostics

Matrix metrics are calculated for every panel feature. Full-HPC mode lazily aggregates `transcripts.parquet`; local subset mode records `NOT_RUN_LOCAL_SUBSET` and does not imply acceptable transcript quality.

In [ ]:
matrix_gene_quality <- summarise_gene_matrix_qc(xenium$counts, REGION_ID, gene_sets)
if (extended_mode == "FULL_HPC") {
  transcript_gene_quality <- summarise_transcript_quality_arrow(file.path(region_dir, "transcripts.parquet"), REGION_ID, extended_config$qv_threshold)
  gene_quality <- combine_gene_quality(matrix_gene_quality, transcript_gene_quality)
} else {
  gene_quality <- matrix_gene_quality
  gene_quality$transcript_rows <- NA_integer_
  gene_quality$mean_qv <- NA_real_
  gene_quality$fraction_q20 <- NA_real_
  gene_quality$represented_codewords <- NA_integer_
  gene_quality$transcript_status <- "NOT_RUN_LOCAL_SUBSET"
}
gene_quality[seq_len(min(12L, nrow(gene_quality))), , drop = FALSE]


## Extended spatial diagnostics

Coordinates test global clustering, an occupied-grid tissue-edge proxy, a kNN dense-aggregate proxy, and candidate hotspot bins. These labels are coordinate evidence only: folds, tears, and other morphology require image review.

In [ ]:
spatial_cells <- assign_spatial_grid(qc$cell_metadata, extended_config$grid_size_um)
spatial_k <- min(as.integer(extended_config$spatial_k), nrow(spatial_cells) - 1L)
spatial_cells$local_density <- calculate_knn_density(spatial_cells, spatial_k, extended_mode)
density_cutoff <- stats::quantile(spatial_cells$local_density, extended_config$dense_quantile, na.rm = TRUE)
spatial_cells$dense_aggregate <- spatial_cells$local_density >= density_cutoff
spatial_global <- test_spatial_flag_clustering(spatial_cells, spatial_k, extended_config$permutations, extended_config$seed, extended_mode)
spatial_global$region_id <- REGION_ID
spatial_edge_density <- summarise_spatial_enrichment(spatial_cells)
spatial_edge_density$region_id <- REGION_ID
spatial_hotspots <- find_spatial_qc_hotspots(spatial_cells, extended_config$permutations, extended_config$min_bin_cells, extended_config$hotspot_fdr, extended_config$seed)
spatial_hotspots$region_id <- rep(REGION_ID, nrow(spatial_hotspots))
manual_review_manifest <- spatial_hotspots[spatial_hotspots$hotspot_status == "MORPHOLOGY_REVIEW_REQUIRED", , drop = FALSE]
if (!nrow(manual_review_manifest)) manual_review_manifest$review_note <- character() else manual_review_manifest$review_note <- "Inspect morphology/image for edge, fold, tear, or dense aggregate context"
list(global = spatial_global, enrichment = spatial_edge_density, candidate_hotspots = manual_review_manifest)


## Extended spatial figures

In [ ]:
extended_plots <- plot_extended_spatial_qc(spatial_cells, spatial_edge_density, spatial_hotspots, REGION_ID)
for (plot in extended_plots) print(plot)


## Evidence-only downstream masks

The raw objects are not modified. `primary_include`, `strict_include`, and `hotspot_sensitivity_include` are retained together so downstream notebooks can select a prespecified analysis without deleting cells. Region 3 hotspot cells remain in primary analysis; Region 4 is sensitivity-only.

In [ ]:
mask_provenance <- paste(RUN_LABEL, REGION_ID, extended_mode, normalizePath(region_dir, winslash = "/", mustWork = TRUE), sep = "|")
downstream_masks <- build_cell_downstream_masks(spatial_cells, spatial_hotspots, mask_provenance)
section_downstream_decision <- build_one_section_downstream_decision(
  downstream_masks, RUN_LABEL, extended_mode, mask_provenance,
  format(Sys.time(), tz = "UTC", usetz = TRUE)
)
section_downstream_decision
with(downstream_masks, c(primary_include = sum(primary_include), strict_include = sum(strict_include), hotspot_sensitivity_include = sum(hotspot_sensitivity_include)))


## Checks

Write every required artifact, then reload the saved sparse object and verify dimensions and cell alignment.

In [ ]:
artifact_paths <- write_section_artifacts(
  project_root = PROJECT_ROOT, output_dir = SECTION_OUTPUT_DIR, region_id = REGION_ID,
  configuration = configuration, manifest = section_manifest, environment = environment,
  inventory = inventory, integrity = integrity, feature_type_summary = feature_type_summary,
  panel_reconciliation = panel_reconciliation, alarms = alarms, qc = qc,
  counts = xenium$counts, features = xenium$features, strict_mode = STRICT_MODE
)
stopifnot(validate_section_artifacts(SECTION_OUTPUT_DIR, REGION_ID))
readiness <- utils::read.delim(file.path(SECTION_OUTPUT_DIR, "section_readiness_gates.tsv"), check.names = FALSE)
readiness


In [ ]:
extended_artifact_paths <- write_extended_section_artifacts(
  project_root = PROJECT_ROOT, output_dir = SECTION_OUTPUT_DIR, region_id = REGION_ID, mode = extended_mode,
  preflight = extended_preflight, cycle_alarm_evidence = cycle_alarm_evidence, gene_quality = gene_quality,
  spatial_global = spatial_global, spatial_edge_density = spatial_edge_density, spatial_hotspots = spatial_hotspots,
  spatial_cells = spatial_cells, manual_review_manifest = manual_review_manifest, plots = extended_plots
)
stopifnot(validate_extended_section_artifacts(SECTION_OUTPUT_DIR, REGION_ID, extended_mode))
extended_reload <- read_extended_section_artifacts(SECTION_OUTPUT_DIR, REGION_ID, extended_mode)
stopifnot(nrow(extended_reload$spatial_cells) == nrow(qc$cell_metadata))
extended_reload$status


In [ ]:
section_mask_path <- write_gz_tsv(downstream_masks, file.path(SECTION_OUTPUT_DIR, "cell_downstream_masks.tsv.gz"), PROJECT_ROOT)
section_decision_path <- write_tsv(section_downstream_decision, file.path(SECTION_OUTPUT_DIR, "section_downstream_decision.tsv"), PROJECT_ROOT)
stopifnot(file.exists(section_mask_path), file.exists(section_decision_path))


## Outputs

All outputs are section-specific. The section notebook creates `cell_downstream_masks.tsv.gz` and `section_downstream_decision.tsv`; the slide summary later freezes cross-section gene tiers and creates the final per-region downstream RDS bundle.

In [ ]:
data.frame(artifact = basename(artifact_paths), path = artifact_paths)[seq_len(min(length(artifact_paths), 20L)), , drop = FALSE]
data.frame(artifact = basename(extended_artifact_paths), path = extended_artifact_paths)
cat("Completed", REGION_ID, "with", nrow(qc$cell_metadata), "cells; zero cells deleted. Extended mode:", extended_mode, "\n")
